# OptiMPa — Concrete Compressive Strength Predictor
### Phase 1: Data Pipeline & Model Training

---

**Project Goal:** Predict concrete compressive strength (MPa) from mix design parameters using a Random Forest Regressor.

**Dataset:** UCI Concrete Compressive Strength — I-Cheng Yeh (1998). 1030 samples.  
Loaded directly via the official `ucimlrepo` library — no CSV files, no GitHub dependencies.

**Features (Civil Engineering → ML):**

| Feature | Unit | CE Significance |
|---|---|---|
| Cement | kg/m³ | Primary binder — drives early strength |
| Blast Furnace Slag | kg/m³ | Latent hydraulic binder, improves long-term strength |
| Fly Ash | kg/m³ | Pozzolanic filler, reduces heat of hydration |
| Water | kg/m³ | w/c ratio controls porosity and strength |
| Superplasticizer | kg/m³ | Reduces water demand without losing workability |
| Coarse Aggregate | kg/m³ | Structural skeleton of the mix |
| Fine Aggregate | kg/m³ | Fills voids between coarse particles |
| Age | days | Hydration progress (28-day standard strength) |

**Target:** Compressive Strength [MPa]

In [ ]:
# ─────────────────────────────────────────────────────────
# 0. IMPORTS & CONFIGURATION
# pip install ucimlrepo  ← only needed once
# ─────────────────────────────────────────────────────────
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

# Aesthetic config for all plots
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='muted')

RANDOM_STATE = 42
os.makedirs('reports', exist_ok=True)
print('Environment ready')

## 1. Data Loading

`fetch_ucirepo(id=165)` calls the official UCI ML Repository API.  
The library caches the response locally after the first download — subsequent runs are instant.

In [ ]:
# ─────────────────────────────────────────────────────────
# 1. LOAD DATASET — ucimlrepo (official UCI Python client)
#
# fetch_ucirepo(id=165) fetches the Concrete Compressive Strength
# dataset directly from the UCI API. No CSV management needed.
# dataset.data.features → X (8 mix-design columns)
# dataset.data.targets  → y (compressive strength in MPa)
# ─────────────────────────────────────────────────────────
dataset = fetch_ucirepo(id=165)

df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)

# Compact column names — consistent with the API contract in Phase 2
df.columns = [
    'cement', 'slag', 'fly_ash', 'water',
    'superplasticizer', 'coarse_agg', 'fine_agg', 'age',
    'strength'   # target
]

print(f'Dataset shape : {df.shape}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# ─────────────────────────────────────────────────────────
# 2a. DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────
print(f'Strength range: {df.strength.min():.1f} - {df.strength.max():.1f} MPa')
df.describe().round(2)

In [ ]:
# ─────────────────────────────────────────────────────────
# 2b. TARGET DISTRIBUTION + ABRAMS' LAW
# Concrete strength typically follows a near-normal distribution
# centred around 35 MPa (standard structural concrete grade).
# Abrams' Law (1919): Strength ~ A / B^(w/c) — higher w/c = lower strength.
# ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['strength'], bins=30, color='#4F86C6', edgecolor='black', alpha=0.8)
axes[0].set_title('Target Distribution — Compressive Strength')
axes[0].set_xlabel('Strength (MPa)')
axes[0].set_ylabel('Count')

# Water-Cement ratio vs Strength (Abrams' Law visualisation)
wc_ratio = df['water'] / df['cement']
axes[1].scatter(wc_ratio, df['strength'], alpha=0.4, color='#F4A261', s=30)
axes[1].set_title("Abrams' Law: w/c Ratio vs Strength")
axes[1].set_xlabel('Water / Cement Ratio')
axes[1].set_ylabel('Strength (MPa)')

plt.tight_layout()
plt.savefig('reports/eda_distributions.png', dpi=150)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────
# 2c. CORRELATION HEATMAP
# Age and cement show highest positive correlation with strength.
# Water shows negative correlation — higher w/c = more porous = weaker.
# ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df.corr(),
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('reports/correlation_heatmap.png', dpi=150)
plt.show()

## 3. Preprocessing

In [ ]:
# ─────────────────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# Using the 8 canonical features that the API will receive.
# The model learns the w/c relationship implicitly — no hand-engineering.
# ─────────────────────────────────────────────────────────
FEATURES = ['cement', 'slag', 'fly_ash', 'water',
            'superplasticizer', 'coarse_agg', 'fine_agg', 'age']
TARGET   = 'strength'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

## 4. Model Training — Random Forest Regressor

> **Why Random Forest?**  
> Concrete strength prediction involves highly **non-linear** interactions (e.g., Abrams' Law, pozzolanic reactions). Linear models fail to capture this. RF handles non-linearity natively, provides **feature importances** for CE interpretation, and is robust to the outlier batches common in real-world concrete test data.

In [ ]:
# ─────────────────────────────────────────────────────────
# 4. TRAIN THE MODEL
# ─────────────────────────────────────────────────────────
model = RandomForestRegressor(
    n_estimators     = 300,    # 300 trees — diminishing returns beyond this
    max_features     = 'sqrt', # sqrt(8)~3 features/split -> decorrelates trees
    min_samples_leaf = 2,      # Prevents overfitting on tiny leaf nodes
    n_jobs           = -1,     # Parallelise across all CPU cores
    random_state     = RANDOM_STATE,
)

model.fit(X_train, y_train)
print('Model trained successfully')

## 5. Evaluation

In [ ]:
# ─────────────────────────────────────────────────────────
# 5a. METRICS
# RMSE in MPa is directly interpretable for Civil Engineers:
# ±5 MPa accuracy is considered acceptable for mix design.
# ─────────────────────────────────────────────────────────
y_pred = model.predict(X_test)

test_r2   = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# 5-fold CV on training data — guards against overfitting
cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)

print(f'Test  R2   : {test_r2:.4f}')
print(f'Test  RMSE : {test_rmse:.4f} MPa  (<- interpret in engineering units)')
print(f'CV R2      : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}  (5-fold)')

In [ ]:
# ─────────────────────────────────────────────────────────
# 5b. VISUALISATION: Predicted vs Actual + Feature Importance
# ─────────────────────────────────────────────────────────
importance_df = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Scatter: Perfect prediction line
axes[0].scatter(y_test, y_pred, alpha=0.6, color='#4F86C6',
                edgecolors='white', linewidth=0.4, s=60)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=1.5, label='Perfect')
axes[0].set_xlabel('Actual Strength (MPa)')
axes[0].set_ylabel('Predicted Strength (MPa)')
axes[0].set_title(f'Predicted vs Actual  |  R2 = {test_r2:.4f}  |  RMSE = {test_rmse:.2f} MPa')
axes[0].legend()

# Feature importance bar chart
axes[1].barh(importance_df['feature'], importance_df['importance'],
             color='#4F86C6', edgecolor='white')
axes[1].set_xlabel('Relative Importance (Gini)')
axes[1].set_title('Feature Importances')
axes[1].invert_yaxis()

plt.suptitle('OptiMPa — Random Forest Evaluation Dashboard', fontsize=15)
plt.tight_layout()
plt.savefig('reports/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Export Model

In [ ]:
# ─────────────────────────────────────────────────────────
# 6. EXPORT — joblib is preferred over pickle for sklearn models
#    because it uses memory-mapped numpy arrays (faster for large forests).
#    The ../api/ path places the artifact directly where FastAPI expects it.
# ─────────────────────────────────────────────────────────
MODEL_PATH = '../api/model.pkl'
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

joblib.dump(model, MODEL_PATH)

size_kb = os.path.getsize(MODEL_PATH) / 1024
print(f'model.pkl exported -> {MODEL_PATH}  ({size_kb:.1f} KB)')
print('\n--- Phase 1 Complete ---')
print('  Next: Phase 2 - FastAPI server in /api/main.py')